In [ ]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

In [ ]:
data = pd.read_csv("fra_perfumes.csv")
data.head()

In [ ]:
def bayesian_avg(R, v, C, m):
    return (v / (v + m)) * R + (m / (v + m)) * C

def clean_text(text):
    if not isinstance(text, str):
        return ""
    return text.replace("[", "").replace("]", "").replace("'", "")

def clean_accords(a):
    if isinstance(a, str):
        return a.replace(",", " ").replace("/", " ")
    return ""

def clean_data(df):
    data = df.copy()

    data.dropna(subset=["Description"], inplace=True)
    data.reset_index(drop=True, inplace=True)

    data["Name"] = [
        b.replace(a, "").strip()
        for a, b in zip(data["Gender"].astype(str), data["Name"].astype(str))
    ]
    data["Brand"] = data["Name"].apply(lambda x: x.split(" ")[-1])
    data["Name"] = [
        b.replace(a, "").strip()
        for a, b in zip(data["Brand"].astype(str), data["Name"].astype(str))
    ]

    data["Description"] = data["Description"].fillna("")
    data["Main Accords"] = data["Main Accords"].fillna("")
    data["Perfumers"] = data["Perfumers"].fillna("")
    data["Gender"] = data["Gender"].fillna("for women and men")
    data["url"] = data["url"].fillna("")

    data["Rating Count"] = (
        data["Rating Count"].astype(str).str.replace(",", "", regex=False).fillna("0").astype(int)
    )
    data["Rating Value"] = data["Rating Value"].fillna(0.0).astype(float)

    gender_map = {
        "for women": "women",
        "for men": "men",
        "for women and men": "unisex",
    }
    data["Gender"] = data["Gender"].replace(gender_map)

    for col in ["Main Accords", "Perfumers"]:
        data[col] = data[col].apply(clean_text)

    data["Main Accords"] = data["Main Accords"].apply(clean_accords)

    data["Description"] = data["Description"].apply(
        lambda x: "".join(x.split(".")[1:]).strip() if "." in x else x.strip()
    )

    C = data["Rating Value"].mean()
    m = data["Rating Count"].quantile(0.75)
    data["Weighted Rating"] = data.apply(
        lambda x: bayesian_avg(x["Rating Value"], x["Rating Count"], C, m), axis=1
    )

    return data[["Name", "Gender", "Main Accords", "Description", "url", "Weighted Rating", "Brand"]]

In [ ]:
df_clean = clean_data(data)
df_clean.to_pickle("data.pkl")
print(f"Cleaned dataset shape: {df_clean.shape}")
df_clean.head()

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
UNISEX_VEC = np.array([1 / np.sqrt(2), 1 / np.sqrt(2)], dtype=np.float32)

GENDER_VECTORS = {
    "men":    np.array([1.0, 0.0], dtype=np.float32),
    "male":   np.array([1.0, 0.0], dtype=np.float32),
    "women":  np.array([0.0, 1.0], dtype=np.float32),
    "female": np.array([0.0, 1.0], dtype=np.float32),
    "unisex": UNISEX_VEC,
}

def gender_to_vec(g):
    g = str(g).strip().lower()
    return GENDER_VECTORS.get(g, UNISEX_VEC)

def expand_gender_vec(gv, dim=384):
    return np.pad(gv, (0, dim - len(gv)), mode="constant").astype(np.float32)

gender_embs = np.vstack([expand_gender_vec(gender_to_vec(g)) for g in df_clean["Gender"]]).astype(np.float32)
print("gender_embs shape:", gender_embs.shape)

In [ ]:
def embed_text(df, col, model):
    texts = df[col].fillna("").tolist()
    embeddings = model.encode(texts, show_progress_bar=True)
    return np.asarray(embeddings, dtype=np.float32)

desc_embs = embed_text(df_clean, "Description", model)
accord_embs = embed_text(df_clean, "Main Accords", model)

In [ ]:
def create_brand_matrix(df):
    brands = sorted(df["Brand"].unique())
    brand_to_idx = {b: i for i, b in enumerate(brands)}

    brand_matrix = np.zeros((len(df), len(brands)), dtype=np.float32)
    for i, brand in enumerate(df["Brand"]):
        brand_matrix[i, brand_to_idx[brand]] = 1.0

    return brand_matrix

def generate_brand_embeddings(df, shape=384):
    brand_matrix = create_brand_matrix(df)
    pca = PCA(n_components=shape)
    brand_embs = pca.fit_transform(brand_matrix)
    return brand_embs.astype(np.float32)

brand_embs = generate_brand_embeddings(df_clean, shape=desc_embs.shape[1])
print("brand_embs shape:", brand_embs.shape)

In [ ]:
desc_embs = normalize(desc_embs).astype(np.float32)
accord_embs = normalize(accord_embs).astype(np.float32)
gender_embs = normalize(gender_embs).astype(np.float32)
brand_embs = normalize(brand_embs).astype(np.float32)

np.save("desc_embs.npy", desc_embs)
np.save("accord_embs.npy", accord_embs)
np.save("gender_embss.npy", gender_embs)
np.save("brand_embs.npy", brand_embs)
print("All embeddings normalized and saved successfully.")